In [ ]:
import struct, time
from collections import deque

import serial
import matplotlib.pyplot as plt

PORT = "COM11"
BAUD = 115200

HDR = b"\xAA\x55"
PKT_LEN = 7

def xor_crc8(data: bytes) -> int:
    c = 0
    for b in data:
        c ^= b
    return c & 0xFF

def parse_stream(buf: bytearray):
    out = []
    while True:
        i = buf.find(HDR)
        if i < 0:
            if len(buf) > 1:
                del buf[:-1]
            break
        if i > 0:
            del buf[:i]
        if len(buf) < PKT_LEN:
            break

        pkt = bytes(buf[:PKT_LEN])
        if xor_crc8(pkt[:6]) != pkt[6]:
            del buf[0:1]
            continue

        del buf[:PKT_LEN]
        meas10, ref10 = struct.unpack_from("<hh", pkt, 2)
        out.append((meas10/10.0, ref10/10.0))
    return out

def main():
    ser = serial.Serial(PORT, BAUD, timeout=0.05)
    time.sleep(0.2)
    ser.reset_input_buffer()

    N = 2000
    t0 = time.time()
    ts = deque(maxlen=N)
    meas = deque(maxlen=N)
    ref  = deque(maxlen=N)

    buf = bytearray()

    plt.ion()
    fig, ax = plt.subplots()
    l1, = ax.plot([], [], label="rpm_meas")
    l2, = ax.plot([], [], label="rpm_ref", ls="--")
    ax.set_xlabel("t, s")
    ax.set_ylabel("rpm")
    ax.grid(True)
    ax.legend()

    try:
        while True:
            chunk = ser.read(256)
            if chunk:
                buf.extend(chunk)
                for m, r in parse_stream(buf):
                    ts.append(time.time() - t0)
                    meas.append(m)
                    ref.append(r)

            if len(ts) > 2:
                l1.set_data(ts, meas)
                l2.set_data(ts, ref)
                ax.relim()
                ax.autoscale_view()
                fig.canvas.draw()
                fig.canvas.flush_events()

            time.sleep(0.03)

    except KeyboardInterrupt:
        pass
    finally:
        ser.close()

if __name__ == "__main__":
    main()
